# MPEC vs. Nested Fixed Point Tutorial

In [1]:
import pyblp
import numpy as np
import pandas as pd

pyblp.options.digits = 2
pyblp.options.verbose = False
pyblp.__version__

'1.2.0'

By default, :meth:`Problem.solve` estimates $\hat{\theta}$ with the nested fixed point (NFP) approach: for every candidate $\theta$, a contraction mapping (described in :ref:`background:The Objective`) inverts the market share equations for $\delta(\theta)$ before the GMM objective and its gradient are evaluated. :ref:`references:Dubé, Fox, and Su (2012)`, building on :ref:`references:Su and Judd (2012)`, propose an alternative MPEC (mathematical program with equilibrium constraints) formulation, described in :ref:`background:MPEC`, that instead treats $\delta$ as a free variable and imposes the market share equations as equality constraints. Su and Judd (2012) show that MPEC and NFP compute the same statistical estimator; MPEC only changes how it is computed.

In this tutorial, we'll re-solve the random coefficients logit problem from the fake cereal data of :ref:`references:Nevo (2000a)`, following the same setup as the Random Coefficients Logit Tutorial, and compare the estimates recovered by pyblp's default NFP approach against those recovered by ``Optimization('mpec-trust-constr')``, pyblp's MPEC implementation using SciPy's [trust-constr](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-trustconstr.html) routine. MPEC currently only supports demand-side estimation with $\hat{\beta}$ fully concentrated out, which is satisfied by this example.

## Loading Data and Configuring the Problem

In [2]:
product_data = pd.read_csv(pyblp.data.NEVO_PRODUCTS_LOCATION)
product_data.head()

,market_ids,city_ids,quarter,product_ids,firm_ids,brand_ids,shares,prices,sugar,mushy,...,demand_instruments10,demand_instruments11,demand_instruments12,demand_instruments13,demand_instruments14,demand_instruments15,demand_instruments16,demand_instruments17,demand_instruments18,demand_instruments19
0,C01Q1,1,1,F1B04,1,4,0.012417,0.072088,2,1,...,2.116358,-0.154708,-0.005796,0.014538,0.126244,0.067345,0.068423,0.034800,0.126346,0.035484
1,C01Q1,1,1,F1B06,1,6,0.007809,0.114178,18,1,...,-7.374091,-0.576412,0.012991,0.076143,0.029736,0.087867,0.110501,0.087784,0.049872,0.072579
2,C01Q1,1,1,F1B07,1,7,0.012995,0.132391,4,1,...,2.187872,-0.207346,0.003509,0.091781,0.163773,0.111881,0.108226,0.086439,0.122347,0.101842
3,C01Q1,1,1,F1B09,1,9,0.005770,0.130344,3,0,...,2.704576,0.040748,-0.003724,0.094732,0.135274,0.088090,0.101767,0.101777,0.110741,0.104332
4,C01Q1,1,1,F1B11,1,11,0.017934,0.154823,12,0,...,1.261242,0.034836,-0.000568,0.102451,0.130640,0.084818,0.101075,0.125169,0.133464,0.121111


In [3]:
X1_formulation = pyblp.Formulation('0 + prices', absorb='C(product_ids)')
X2_formulation = pyblp.Formulation('1 + prices + sugar + mushy')
product_formulations = (X1_formulation, X2_formulation)
product_formulations

(prices + Absorb[C(product_ids)], 1 + prices + sugar + mushy)

As in the Random Coefficients Logit Tutorial, we'll use Monte Carlo integration with 50 simulated individuals per market. We leave out demographic interactions to keep this example's runtime manageable; the same comparison applies to problems with demographics.

In [4]:
mc_integration = pyblp.Integration('monte_carlo', size=50, specification_options={'seed': 0})
mc_problem = pyblp.Problem(product_formulations, product_data, integration=mc_integration)
mc_problem

Dimensions:
 T    N     F    I     K1    K2    MD    ED 
---  ----  ---  ----  ----  ----  ----  ----
94   2256   5   4700   1     4     20    1  

Formulations:
       Column Indices:           0       1       2      3  
-----------------------------  ------  ------  -----  -----
 X1: Linear Characteristics    prices                      
X2: Nonlinear Characteristics    1     prices  sugar  mushy

## Solving with the Nested Fixed Point Approach

We'll use the same non-default BFGS configuration as the Random Coefficients Logit Tutorial, starting $\Sigma_0$ at a matrix of ones so that the full covariance matrix for random tastes is estimated.

In [5]:
bfgs = pyblp.Optimization('bfgs', {'gtol': 1e-4})
nfp_results = mc_problem.solve(sigma=np.ones((4, 4)), optimization=bfgs)
nfp_results

Problem Results Summary:
GMM   Objective  Gradient      Hessian         Hessian     Clipped  Weighting Matrix  Covariance Matrix
Step    Value      Norm    Min Eigenvalue  Max Eigenvalue  Shares   Condition Number  Condition Number 
----  ---------  --------  --------------  --------------  -------  ----------------  -----------------
 2    +1.5E+02   +8.7E-05     +8.5E-02        +6.5E+03        0         +5.2E+07          +8.3E+05     

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective   Fixed Point  Contraction
   Time      Converged   Iterations   Evaluations  Iterations   Evaluations
-----------  ---------  ------------  -----------  -----------  -----------
 00:00:21       Yes          58           75          88245       270904   

Nonlinear Coefficient Estimates (Robust SEs in Parentheses):
Sigma:      1         prices      sugar       mushy     |  Sigma Squared:      1         prices      sugar       mushy   
------  ----------  ----------  ----------  ---

## Solving with MPEC

We configure MPEC with ``Optimization('mpec-trust-constr')`` and, to start, use the exact same starting values as the NFP problem above: $\Sigma_0$ equal to a matrix of ones, and the default starting $\delta$ (the closed-form solution to the plain logit model).

In [6]:
mpec_cold_results = mc_problem.solve(sigma=np.ones((4, 4)), optimization=pyblp.Optimization('mpec-trust-constr'))
mpec_cold_results

Problem Results Summary:
GMM   Objective    Projected    Reduced Hessian  Reduced Hessian  Clipped  Weighting Matrix  Covariance Matrix
Step    Value    Gradient Norm  Min Eigenvalue   Max Eigenvalue   Shares   Condition Number  Condition Number 
----  ---------  -------------  ---------------  ---------------  -------  ----------------  -----------------
 2    +1.3E+02     +1.8E-06        +6.9E-02         +5.6E+03         0         +5.2E+07          +6.8E+05     

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective   Fixed Point  Contraction
   Time      Converged   Iterations   Evaluations  Iterations   Evaluations
-----------  ---------  ------------  -----------  -----------  -----------
 00:04:25       No          1212         1283          538         1809    

Nonlinear Coefficient Estimates (Robust SEs in Parentheses):
Sigma:      1         prices      sugar       mushy     |  Sigma Squared:      1         prices      sugar       mushy   
------  ---------- 

From this starting point, MPEC's optimizer reports that it did **not** converge ("Optimizer Converged: No" above) after over a thousand iterations, and settles on a noticeably different point than NFP did. Like NFP, MPEC solves a nonconvex problem, so its behavior can depend on starting values and on the particular nonlinear solver used. :ref:`references:Dubé, Fox, and Su (2012)` benchmark MPEC using [Artleys Knitro](https://www.artelys.com/solvers/knitro/) (also usable in pyblp via ``Optimization('mpec-knitro')``, if a license is available), which tends to be more robust for these problems than SciPy's general-purpose ``trust-constr`` routine used here. In practice, it is a good idea to try multiple starting values with either approach, as recommended throughout pyblp's documentation.

To confirm that MPEC and NFP recover the *same* estimator when they converge to the same point, we'll re-solve with MPEC, this time starting from the NFP estimates above.

In [7]:
mpec_warm_results = mc_problem.solve(
    sigma=nfp_results.sigma,
    optimization=pyblp.Optimization('mpec-trust-constr'),
    delta=nfp_results.delta,
)
mpec_warm_results

Problem Results Summary:
GMM   Objective    Projected    Reduced Hessian  Reduced Hessian  Clipped  Weighting Matrix  Covariance Matrix
Step    Value    Gradient Norm  Min Eigenvalue   Max Eigenvalue   Shares   Condition Number  Condition Number 
----  ---------  -------------  ---------------  ---------------  -------  ----------------  -----------------
 2    +1.5E+02     +1.5E-06        +8.5E-02         +6.5E+03         0         +5.2E+07          +8.3E+05     

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective   Fixed Point  Contraction
   Time      Converged   Iterations   Evaluations  Iterations   Evaluations
-----------  ---------  ------------  -----------  -----------  -----------
 00:01:30       Yes         263           343           0           193    

Nonlinear Coefficient Estimates (Robust SEs in Parentheses):
Sigma:      1         prices      sugar       mushy     |  Sigma Squared:      1         prices      sugar       mushy   
------  ---------- 

## Comparing Estimates

In [8]:
comparison = pd.DataFrame({
    'NFP': [
        float(nfp_results.objective),
        float(nfp_results.sigma[0, 0]),
        float(nfp_results.sigma[1, 1]),
        float(nfp_results.sigma[2, 2]),
        float(nfp_results.sigma[3, 3]),
        float(nfp_results.beta[0, 0]),
    ],
    'MPEC (cold start)': [
        float(mpec_cold_results.objective),
        float(mpec_cold_results.sigma[0, 0]),
        float(mpec_cold_results.sigma[1, 1]),
        float(mpec_cold_results.sigma[2, 2]),
        float(mpec_cold_results.sigma[3, 3]),
        float(mpec_cold_results.beta[0, 0]),
    ],
    'MPEC (warm start)': [
        float(mpec_warm_results.objective),
        float(mpec_warm_results.sigma[0, 0]),
        float(mpec_warm_results.sigma[1, 1]),
        float(mpec_warm_results.sigma[2, 2]),
        float(mpec_warm_results.sigma[3, 3]),
        float(mpec_warm_results.beta[0, 0]),
    ],
}, index=['objective', 'sigma[0, 0]', 'sigma[1, 1]', 'sigma[2, 2]', 'sigma[3, 3]', 'alpha (prices)'])
comparison

,NFP,MPEC (cold start),MPEC (warm start)
objective,148.366509,127.609112,148.366512
"sigma[0, 0]",1.207566,1.293091,1.207566
"sigma[1, 1]",8.423600,8.014956,8.423601
"sigma[2, 2]",0.037834,0.046353,0.037834
"sigma[3, 3]",0.480005,0.519393,0.480005
alpha (prices),-31.373498,-31.527988,-31.373498


When warm-started from the same point, MPEC and NFP agree to several decimal places on the objective value, $\hat{\Sigma}$, and $\hat{\alpha}$, consistent with :ref:`references:Su and Judd (2012)`'s result that the two approaches compute the same estimator. We can also check that their standard errors agree.

In [9]:
print('max |sigma difference| =', np.nanmax(np.abs(nfp_results.sigma - mpec_warm_results.sigma)))
print('max |sigma_se difference| =', np.nanmax(np.abs(nfp_results.sigma_se - mpec_warm_results.sigma_se)))
print('max |beta difference| =', np.nanmax(np.abs(nfp_results.beta - mpec_warm_results.beta)))
print('max |beta_se difference| =', np.nanmax(np.abs(nfp_results.beta_se - mpec_warm_results.beta_se)))
print('max |xi difference| =', np.nanmax(np.abs(nfp_results.xi - mpec_warm_results.xi)))

max |sigma difference| = 4.0863997252671425e-06
max |sigma_se difference| = 6.608362109972177e-06
max |beta difference| = 4.702824298874475e-07
max |beta_se difference| = 1.7943343859627703e-06
max |xi difference| = 1.128062315558509e-06


The remaining differences are on the order of the two solvers' default optimization tolerances, not a meaningful economic difference. The cold-started MPEC estimates above, in contrast, differ substantially from both, illustrating that the choice between NFP and MPEC is a purely computational one: both target the same GMM estimator, and, as with any nonconvex estimation problem, getting reliable estimates from either one requires reasonable starting values (and, ideally, checking multiple of them).